# Notebook 03 — Train Risk Scorer → .pkl

Trains a Random Forest risk/trust scorer from your synthetic transaction dataset.

In [ ]:
import os, json, hashlib, time
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, precision_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
import joblib

EXPORTS_DIR = Path(os.getenv('EXPORTS_DIR', r'C:\Users\MJ\Desktop\Agric\jupyter\exports\models'))
TX_CSV = Path(os.getenv('TRANSACTION_DATASET_CSV', r'C:\Users\MJ\Desktop\Agric\jupyter\data\transactions\synthetic_transactions.csv'))
MODEL_VERSION = os.getenv('MODEL_VERSION', 'v1')
MIN_ROWS = int(os.getenv('MIN_RISK_ROWS', '100'))
MIN_PRECISION = float(os.getenv('MIN_RISK_PRECISION', '0.70'))
ALLOW_SYNTHETIC_DATA = os.getenv('ALLOW_SYNTHETIC_DATA', 'false').lower() == 'true'
EXPORTS_DIR.mkdir(parents=True, exist_ok=True)

def sha256_file(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

print('TX_CSV:', TX_CSV)
print('EXPORTS_DIR:', EXPORTS_DIR)

In [ ]:
if TX_CSV.exists():
    df = pd.read_csv(TX_CSV)
    dataset_source = str(TX_CSV)
elif ALLOW_SYNTHETIC_DATA:
    rng = np.random.default_rng(42)
    n = 10000
    df = pd.DataFrame({
        'amount': rng.lognormal(5, 1.2, n),
        'status': rng.choice(['completed', 'failed', 'cancelled'], n, p=[0.85, 0.10, 0.05]),
        'dispute_raised': rng.choice([0, 1], n, p=[0.95, 0.05]),
        'farmer_trust': rng.uniform(20, 100, n),
        'buyer_trust': rng.uniform(20, 100, n),
        'account_age_days': rng.integers(1, 1200, n),
        'created_at': pd.date_range('2023-01-01', periods=n, freq='h'),
    })
    dataset_source = 'synthetic_smoke_test'
else:
    raise FileNotFoundError(f'Risk dataset not found: {TX_CSV}')

if len(df) < MIN_ROWS:
    raise ValueError(f'Risk dataset has {len(df)} rows, minimum required is {MIN_ROWS}')

if 'status' not in df.columns:
    raise ValueError('Risk dataset requires a status column')
if 'dispute_raised' not in df.columns:
    df['dispute_raised'] = 0
if 'account_age_days' not in df.columns:
    df['account_age_days'] = 180

df['is_success'] = df['status'].astype(str).str.lower().isin(['completed', 'success', 'settled']).astype(int)
df['is_risky'] = ((df['is_success'] == 0) | (df['dispute_raised'].astype(int) == 1)).astype(int)
df['log_amount'] = np.log1p(pd.to_numeric(df.get('amount', 0), errors='coerce').fillna(0))
df['farmer_trust'] = pd.to_numeric(df.get('farmer_trust', 50), errors='coerce').fillna(50)
df['buyer_trust'] = pd.to_numeric(df.get('buyer_trust', 50), errors='coerce').fillna(50)
df['account_age_days'] = pd.to_numeric(df['account_age_days'], errors='coerce').fillna(180)

FEATURES = ['log_amount', 'dispute_raised', 'farmer_trust', 'buyer_trust', 'account_age_days']
TARGET = 'is_risky'
X = df[FEATURES].values.astype(np.float32)
y = df[TARGET].values.astype(int)
print('Dataset source:', dataset_source)
print('Rows:', len(df), 'Risk rate:', y.mean())

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y if len(set(y)) > 1 else None)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
model = RandomForestClassifier(n_estimators=250, max_depth=12, class_weight='balanced', random_state=42, n_jobs=-1)
model.fit(X_train_s, y_train)
pred = model.predict(X_val_s)
precision = precision_score(y_val, pred, zero_division=0)
try:
    auc = roc_auc_score(y_val, model.predict_proba(X_val_s)[:, 1])
except Exception:
    auc = 0.0
print(classification_report(y_val, pred, zero_division=0))
print('Precision:', precision, 'AUC:', auc)
assert precision >= MIN_PRECISION, f'Risk precision {precision:.2%} below threshold {MIN_PRECISION:.0%}'

In [ ]:
model_path = EXPORTS_DIR / 'risk_scorer_v1.pkl'
scaler_path = EXPORTS_DIR / 'risk_scaler_v1.pkl'
joblib.dump(model, model_path, compress=3)
joblib.dump(scaler, scaler_path, compress=3)
meta = {'model': 'risk_scorer_v1', 'version': MODEL_VERSION, 'format': 'pkl', 'source_notebook': '03_train_risk_scorer.ipynb', 'dataset_source': dataset_source, 'dataset_rows': int(len(df)), 'features': FEATURES, 'target': TARGET, 'precision': round(float(precision), 4), 'roc_auc': round(float(auc), 4), 'sha256': sha256_file(model_path), 'scaler_sha256': sha256_file(scaler_path), 'created_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}
(EXPORTS_DIR / 'risk_scorer_metadata.json').write_text(json.dumps(meta, indent=2))
print(json.dumps(meta, indent=2))